# Auxhead-Lite Modal Run Monitor

This notebook is set up to inspect the `tom-auxhead-lite-runs-v2-20260408` Modal volume created by the current Variant 2 Modal launcher.

Use the cells below to:
- discover finished and in-progress seed runs,
- read `run_summary.json`, `stdout.log`, and `stderr.log`,
- compare seed 7 and seed 11 metrics for a chosen target total,
- plot the saved learning curve from the volume.
- revisit the initial 800-episode seed 7 vs seed 11 analysis figures.


In [ ]:
import csv
import io
import json
from pathlib import PurePosixPath

import matplotlib.pyplot as plt
import modal

VOLUME_NAME = "tom-auxhead-lite-runs-v2-20260408"
RUN_ROOT = PurePosixPath("auxhead-lite")
REMOTE_OUTPUT_PREFIX = PurePosixPath("/root/outputs")

# Client-side volume access: do not call volume.reload() here.
volume = modal.Volume.from_name(VOLUME_NAME)


def _entry_path(entry):
    return PurePosixPath(getattr(entry, "path", str(entry)))


def list_paths(prefix: PurePosixPath = RUN_ROOT):
    return sorted(_entry_path(entry) for entry in volume.iterdir(str(prefix), recursive=True))


def read_text(path: str | PurePosixPath) -> str:
    data = b"".join(volume.read_file(str(path)))
    return data.decode("utf-8")


def tail_text(path: str | PurePosixPath, lines: int = 40) -> str:
    text = read_text(path)
    return "\n".join(text.splitlines()[-lines:])


def remote_to_volume_path(remote_path: str) -> PurePosixPath:
    remote = PurePosixPath(remote_path)
    if remote.is_absolute():
        return remote.relative_to(REMOTE_OUTPUT_PREFIX)
    return remote


all_paths = list_paths()
summary_paths = [path for path in all_paths if path.name == "run_summary.json"]
print(f"volume={VOLUME_NAME}")
print(f"paths_found={len(all_paths)}")
print(f"run_summaries_found={len(summary_paths)}")
summary_paths[:10]


In [ ]:
def load_summaries():
    rows = []
    for path in list_paths():
        if path.name != "run_summary.json":
            continue
        payload = json.loads(read_text(path))
        eval_metrics = payload.get("eval_metrics") or {}
        rows.append(
            {
                "seed": payload.get("seed"),
                "target_total_episodes": payload.get("target_total_episodes"),
                "additional_train_episodes": payload.get("additional_train_episodes"),
                "initial_checkpoint_train_episodes": payload.get("initial_checkpoint_train_episodes"),
                "returncode": payload.get("returncode"),
                "path": str(path),
                "output_dir": payload.get("output_dir"),
                "ToMCoordScore": eval_metrics.get("ToMCoordScore"),
                "SuccessRate": eval_metrics.get("SuccessRate"),
                "CollisionRate": eval_metrics.get("CollisionRate"),
                "DeadlockRate": eval_metrics.get("DeadlockRate"),
                "IntentionPredictionF1": eval_metrics.get("IntentionPredictionF1"),
                "StrategySwitchAccuracy": eval_metrics.get("StrategySwitchAccuracy"),
                "AmbiguityEfficiency": eval_metrics.get("AmbiguityEfficiency"),
                "AverageDelay": eval_metrics.get("AverageDelay"),
            }
        )
    return sorted(rows, key=lambda row: (row["target_total_episodes"], row["seed"]))


def load_status_rows():
    rows = {}
    for path in list_paths():
        if path.name not in {"progress.json", "run_status.json", "run_summary.json"}:
            continue
        parts = path.parts
        if len(parts) < 4 or not parts[1].startswith("seed") or not parts[2].startswith("target-"):
            continue

        seed = int(parts[1].removeprefix("seed"))
        target_total_episodes = int(parts[2].removeprefix("target-"))
        key = (seed, target_total_episodes)
        row = rows.setdefault(
            key,
            {
                "seed": seed,
                "target_total_episodes": target_total_episodes,
                "state": None,
                "completed_total_episodes": None,
                "remaining_total_episodes": None,
                "completed_additional_episodes": None,
                "is_final": None,
                "returncode": None,
                "path": str(path.parent),
                "ToMCoordScore": None,
            },
        )

        payload = json.loads(read_text(path))
        if path.name == "run_status.json":
            row["state"] = payload.get("state")
        elif path.name == "progress.json":
            row["completed_total_episodes"] = payload.get("completed_total_episodes")
            row["remaining_total_episodes"] = payload.get("remaining_total_episodes")
            row["completed_additional_episodes"] = payload.get("completed_additional_episodes")
            row["is_final"] = payload.get("is_final")
        elif path.name == "run_summary.json":
            eval_metrics = payload.get("eval_metrics") or {}
            row["returncode"] = payload.get("returncode")
            row["ToMCoordScore"] = eval_metrics.get("ToMCoordScore")
            if row["state"] is None:
                row["state"] = "completed" if payload.get("returncode") == 0 else "failed"

    return sorted(rows.values(), key=lambda row: (row["target_total_episodes"], row["seed"]))


def latest_target_total(seed: int | None = None) -> int:
    targets = []
    for row in load_summaries():
        if seed is None or row["seed"] == seed:
            targets.append(row["target_total_episodes"])
    if not targets:
        raise ValueError("No run summaries found in the Modal volume yet.")
    return max(targets)

summaries = load_summaries()
summaries


In [ ]:
SEED = 7
TARGET_TOTAL_EPISODES = latest_target_total(seed=SEED)

run_dir = RUN_ROOT / f"seed{SEED}" / f"target-{TARGET_TOTAL_EPISODES}"
stdout_path = run_dir / "stdout.log"
stderr_path = run_dir / "stderr.log"
summary_path = run_dir / "run_summary.json"

print(f"run_dir={run_dir}")
print("\n=== summary ===")
print(read_text(summary_path))
print("\n=== stdout tail ===")
print(tail_text(stdout_path, lines=60))
print("\n=== stderr tail ===")
print(tail_text(stderr_path, lines=60))


In [ ]:
summaries = load_summaries()
status_rows = load_status_rows()
print(f"completed_summaries={len(summaries)}")
print(f"status_rows={len(status_rows)}")
status_rows

# Completed runs only:

TARGET_TOTAL_EPISODES = latest_target_total()

comparison = [
    row
    for row in summaries
    if row["target_total_episodes"] == TARGET_TOTAL_EPISODES
]

print(f"latest_completed_target={TARGET_TOTAL_EPISODES}")
comparison


In [ ]:
SEED = 7
TARGET_TOTAL_EPISODES = latest_target_total(seed=SEED)

summary = json.loads(read_text(RUN_ROOT / f"seed{SEED}" / f"target-{TARGET_TOTAL_EPISODES}" / "run_summary.json"))
curve_path = remote_to_volume_path(summary["learning_curve_csv"])
curve_rows = list(csv.DictReader(io.StringIO(read_text(curve_path))))

episodes = [int(float(row["episode"])) for row in curve_rows]
reward_ma_10 = [float(row["reward_ma_10"]) for row in curve_rows]
reward_sum = [float(row["reward_sum"]) for row in curve_rows]

plt.figure(figsize=(10, 4))
plt.plot(episodes, reward_ma_10, label="reward_ma_10", linewidth=2)
plt.plot(episodes, reward_sum, label="reward_sum", alpha=0.35)
plt.title(f"auxhead-lite warm-start seed {SEED} target {TARGET_TOTAL_EPISODES}")
plt.xlabel("additional training episode")
plt.ylabel("reward")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
LATEST_TARGET_TOTAL = latest_target_total()
LATEST_TARGET_TOTAL


## Initial 800-Run Comparison

These cells display the original seed 7 vs seed 11 analysis figures from the first local 800-episode auxhead-lite comparison. The PNG artifacts are now stored in-repo under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/initial-800-run-analysis`.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

INITIAL_800_ANALYSIS_ROOT = Path("/Users/stephenbeale/Projects/ToM AI Research Team/modal/initial-800-run-analysis")

INITIAL_800_FIGURES = [
    ("Belief transitions (counts)", "belief-change-heatmap.png"),
    ("Belief transitions (normalized)", "normalised transitions heatmap.png"),
    ("Normalized transition delta (seed11 - seed7)", "normalised-transition-delta-seed-7-11.png"),
    ("Mean belief confidence by step index", "mean-belief-confidence-by-seed.png"),
    ("Outcome rate by seed", "outcome-rate-by-seed.png"),
    ("Action distribution by seed", "action-distribution-by-seed.png"),
]

for title, filename in INITIAL_800_FIGURES:
    figure_path = INITIAL_800_ANALYSIS_ROOT / filename
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_path), width=1100))


## Local 130k Comparison

These cells read the in-project local Modal outputs under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-130k-modal-results` and show the generated verdict/charts from `scripts/modal_130k_report.py`.


In [ ]:
from pathlib import Path
import json
from IPython.display import SVG, Markdown, display

LOCAL_RESULTS_ROOT = Path("/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-130k-modal-results")
LOCAL_REPORTS_ROOT = LOCAL_RESULTS_ROOT / "reports"

display(Markdown((LOCAL_REPORTS_ROOT / "modal_130k_verdict.md").read_text()))
summary_json = json.loads((LOCAL_REPORTS_ROOT / "modal_130k_summary.json").read_text())
summary_json["verdicts"]


In [ ]:
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "metric_comparison.svg")))
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "curve_overlay.svg")))
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "family_outcome_grid.svg")))


## Local 140k Comparison

These cells read the completed in-project 140k report pack under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-140k-modal-results/reports`.


In [ ]:
from pathlib import Path
import json
from IPython.display import SVG, Markdown, display

LOCAL_140_RESULTS_ROOT = Path("/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-140k-modal-results")
LOCAL_140_REPORTS_ROOT = LOCAL_140_RESULTS_ROOT / "reports"

display(Markdown((LOCAL_140_REPORTS_ROOT / "modal_longrun_verdict.md").read_text()))
longrun_summary_json = json.loads((LOCAL_140_REPORTS_ROOT / "modal_longrun_summary.json").read_text())
longrun_summary_json["verdicts"]


In [ ]:
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_metric_lines.svg")))
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_curve_overlay.svg")))
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_family_success.svg")))
